# 10 - P00 vs P01 vs P02 vs P03 vs P04

Comparacion de los cinco preprocesamientos de `50. preprocesamiento` usando un unico arbol fijo: el XGBoost baseline de `i3_arboles` que usamos como referencia estable.

La comparacion usa el split ya generado en `data/preprocesor/` y evalua principalmente `AMS` ponderado sobre validacion. Para `P03` se entrena un modelo por grupo (`G0`, `G1`, `G2`) y luego se combinan los scores para calcular un AMS global comparable.

## Imports y configuracion

In [1]:
from __future__ import annotations

import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from xgboost import XGBClassifier


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "roadmap.md").exists() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("No se pudo localizar la raiz del proyecto.")


project_root = find_project_root()
preprocessor_root = project_root / "data" / "preprocesor"
results_dir = project_root / "50. preprocesamiento" / "results"
results_dir.mkdir(parents=True, exist_ok=True)

SEED = 42
XGB_BASELINE_PARAMS = {
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "random_state": SEED,
    "tree_method": "hist",
    "n_jobs": -1,
    "missing": np.nan,
    "learning_rate": 0.05,
    "n_estimators": 400,
    "max_depth": 6,
    "scale_pos_weight": 1.5346,
    "subsample": 0.9,
    "colsample_bytree": 0.9,
    "min_child_weight": 1,
    "gamma": 0.0,
    "reg_lambda": 1.0,
    "reg_alpha": 0.0,
    "max_delta_step": 0,
}

GLOBAL_PIPELINES = {
    "P00": preprocessor_root / "P00_global_standard",
    "P01": preprocessor_root / "P01_jet_onehot_standard",
    "P02": preprocessor_root / "P02_jet_onehot_robust",
    "P04": preprocessor_root / "P04_feature_engineering_robust",
}
P03_DIR = preprocessor_root / "P03_by_jet_groups"

print(f"project_root={project_root}")
print(f"preprocessor_root={preprocessor_root}")


project_root=/home/santig14/fing/taa/1_TallerAA
preprocessor_root=/home/santig14/fing/taa/1_TallerAA/data/preprocesor


## Utilidades AMS y evaluacion

In [2]:
def ams_score(signal_weight: np.ndarray | float, background_weight: np.ndarray | float, br: float = 10.0) -> np.ndarray:
    signal_arr = np.asarray(signal_weight, dtype=np.float64)
    background_arr = np.asarray(background_weight, dtype=np.float64)
    radicand = 2.0 * (
        (signal_arr + background_arr + br) * np.log1p(signal_arr / (background_arr + br)) - signal_arr
    )
    return np.sqrt(np.clip(radicand, a_min=0.0, a_max=None))


def find_best_ams_threshold(y_true: pd.Series | np.ndarray, y_scores: np.ndarray, weights: pd.Series | np.ndarray) -> dict[str, float | int]:
    y_array = np.asarray(y_true, dtype=np.int8)
    score_array = np.asarray(y_scores, dtype=np.float64)
    weight_array = np.asarray(weights, dtype=np.float64)

    order = np.argsort(score_array)[::-1]
    y_sorted = y_array[order]
    score_sorted = score_array[order]
    weight_sorted = weight_array[order]

    signal_cumsum = np.cumsum(np.where(y_sorted == 1, weight_sorted, 0.0))
    background_cumsum = np.cumsum(np.where(y_sorted == 0, weight_sorted, 0.0))
    candidate_mask = np.r_[score_sorted[1:] != score_sorted[:-1], True]
    candidate_indices = np.flatnonzero(candidate_mask)
    ams_values = ams_score(signal_cumsum[candidate_indices], background_cumsum[candidate_indices])

    best_position = int(np.argmax(ams_values))
    best_idx = int(candidate_indices[best_position])
    threshold = float(score_sorted[best_idx])
    y_pred = (score_array >= threshold).astype(np.int8)

    return {
        "best_ams": float(ams_values[best_position]),
        "best_threshold": threshold,
        "signal_weight": float(signal_cumsum[best_idx]),
        "background_weight": float(background_cumsum[best_idx]),
        "predicted_signal_count": int(y_pred.sum()),
        "predicted_background_count": int(len(y_pred) - y_pred.sum()),
        "accuracy_at_ams": float(accuracy_score(y_array, y_pred)),
        "precision_at_ams": float(precision_score(y_array, y_pred, zero_division=0)),
        "recall_at_ams": float(recall_score(y_array, y_pred, zero_division=0)),
        "f1_at_ams": float(f1_score(y_array, y_pred, zero_division=0)),
    }


def evaluate_scores(pipeline_id: str, y_valid: pd.Series, valid_scores: np.ndarray, valid_weight: pd.Series, fit_seconds: float, feature_count: int) -> dict[str, float | int | str]:
    default_pred = (valid_scores >= 0.5).astype(np.int8)
    ams_metrics = find_best_ams_threshold(y_valid, valid_scores, valid_weight)
    return {
        "pipeline": pipeline_id,
        "feature_count": int(feature_count),
        "valid_rows": int(len(y_valid)),
        **ams_metrics,
        "roc_auc": float(roc_auc_score(y_valid, valid_scores)),
        "default_accuracy": float(accuracy_score(y_valid, default_pred)),
        "default_precision": float(precision_score(y_valid, default_pred, zero_division=0)),
        "default_recall": float(recall_score(y_valid, default_pred, zero_division=0)),
        "default_f1": float(f1_score(y_valid, default_pred, zero_division=0)),
        "fit_seconds": float(fit_seconds),
    }


def read_labels_and_weights(path: Path) -> tuple[pd.Series, pd.Series]:
    y_valid = pd.read_csv(path / "y_valid.csv")["Label"].astype(np.int8)
    valid_weight = pd.read_csv(path / "valid_weight.csv")["Weight"].astype(np.float64)
    return y_valid, valid_weight


## Entrenamiento de pipelines globales

In [3]:
rows: list[dict[str, float | int | str]] = []
global_scores: dict[str, np.ndarray] = {}

for pipeline_id, pipeline_dir in GLOBAL_PIPELINES.items():
    print(f"Entrenando {pipeline_id} desde {pipeline_dir.name}...")
    X_train = pd.read_csv(pipeline_dir / "X_train_processed.csv").astype(np.float32)
    X_valid = pd.read_csv(pipeline_dir / "X_valid_processed.csv").astype(np.float32)
    y_train = pd.read_csv(pipeline_dir / "y_train.csv")["Label"].astype(np.int8)
    y_valid, valid_weight = read_labels_and_weights(pipeline_dir)

    model = XGBClassifier(**XGB_BASELINE_PARAMS)
    start = time.perf_counter()
    model.fit(X_train, y_train)
    fit_seconds = time.perf_counter() - start

    valid_scores = model.predict_proba(X_valid)[:, 1]
    global_scores[pipeline_id] = valid_scores
    row = evaluate_scores(pipeline_id, y_valid, valid_scores, valid_weight, fit_seconds, X_train.shape[1])
    rows.append(row)
    print(f"{pipeline_id}: AMS={row['best_ams']:.6f}, AUC={row['roc_auc']:.6f}, threshold={row['best_threshold']:.6f}")


Entrenando P00 desde P00_global_standard...
P00: AMS=1.088267, AUC=0.908913, threshold=0.887274
Entrenando P01 desde P01_jet_onehot_standard...
P01: AMS=1.090579, AUC=0.909002, threshold=0.881231
Entrenando P02 desde P02_jet_onehot_robust...
P02: AMS=1.090579, AUC=0.909002, threshold=0.881231
Entrenando P04 desde P04_feature_engineering_robust...
P04: AMS=1.112924, AUC=0.910205, threshold=0.877203


## Entrenamiento de P03 por grupos de jets

In [4]:
p03_group_rows: list[dict[str, float | int | str]] = []
p03_y_parts: list[pd.Series] = []
p03_weight_parts: list[pd.Series] = []
p03_score_parts: list[np.ndarray] = []
p03_fit_seconds = 0.0
p03_feature_counts: list[int] = []

for group_id in ["G0", "G1", "G2"]:
    group_dir = P03_DIR / group_id
    print(f"Entrenando P03/{group_id}...")
    X_train = pd.read_csv(group_dir / "X_train_processed.csv").astype(np.float32)
    X_valid = pd.read_csv(group_dir / "X_valid_processed.csv").astype(np.float32)
    y_train = pd.read_csv(group_dir / "y_train.csv")["Label"].astype(np.int8)
    y_valid, valid_weight = read_labels_and_weights(group_dir)

    model = XGBClassifier(**XGB_BASELINE_PARAMS)
    start = time.perf_counter()
    model.fit(X_train, y_train)
    fit_seconds = time.perf_counter() - start

    valid_scores = model.predict_proba(X_valid)[:, 1]
    p03_fit_seconds += fit_seconds
    p03_feature_counts.append(int(X_train.shape[1]))
    p03_y_parts.append(y_valid)
    p03_weight_parts.append(valid_weight)
    p03_score_parts.append(valid_scores)

    group_row = evaluate_scores(f"P03_{group_id}", y_valid, valid_scores, valid_weight, fit_seconds, X_train.shape[1])
    p03_group_rows.append(group_row)
    print(f"P03/{group_id}: AMS={group_row['best_ams']:.6f}, AUC={group_row['roc_auc']:.6f}, threshold={group_row['best_threshold']:.6f}")

p03_y_valid = pd.concat(p03_y_parts, ignore_index=True)
p03_valid_weight = pd.concat(p03_weight_parts, ignore_index=True)
p03_valid_scores = np.concatenate(p03_score_parts)
p03_row = evaluate_scores(
    "P03_by_jet_groups",
    p03_y_valid,
    p03_valid_scores,
    p03_valid_weight,
    p03_fit_seconds,
    int(sum(p03_feature_counts)),
)
rows.append(p03_row)
print(f"P03 global: AMS={p03_row['best_ams']:.6f}, AUC={p03_row['roc_auc']:.6f}, threshold={p03_row['best_threshold']:.6f}")


Entrenando P03/G0...
P03/G0: AMS=0.738753, AUC=0.891660, threshold=0.855209
Entrenando P03/G1...
P03/G1: AMS=0.537608, AUC=0.890635, threshold=0.916172
Entrenando P03/G2...
P03/G2: AMS=0.607209, AUC=0.929319, threshold=0.888716
P03 global: AMS=1.069744, AUC=0.907901, threshold=0.874451


## Comparacion final

In [5]:
comparison = pd.DataFrame(rows).sort_values("best_ams", ascending=False).reset_index(drop=True)
comparison_path = results_dir / "10_p0vsp1vsp2vsp3vsp4_comparison.csv"
comparison.to_csv(comparison_path, index=False)

columns = [
    "pipeline",
    "best_ams",
    "roc_auc",
    "best_threshold",
    "precision_at_ams",
    "recall_at_ams",
    "f1_at_ams",
    "predicted_signal_count",
    "feature_count",
    "fit_seconds",
]
display(comparison[columns].round(6))
print(f"Resultados guardados en: {comparison_path}")


,pipeline,best_ams,roc_auc,best_threshold,precision_at_ams,recall_at_ams,f1_at_ams,predicted_signal_count,feature_count,fit_seconds
0,P04,1.112924,0.910205,0.877203,0.928407,0.423836,0.581984,3911,67,5.150450
1,P02,1.090579,0.909002,0.881231,0.928590,0.409828,0.568675,3781,44,3.552174
2,P01,1.090579,0.909002,0.881231,0.928590,0.409828,0.568675,3781,44,3.778184
3,P00,1.088267,0.908913,0.887274,0.932273,0.396872,0.556738,3647,41,6.375640
4,P03_by_jet_groups,1.069744,0.907901,0.874451,0.919217,0.438310,0.593582,4085,74,4.530441


Resultados guardados en: /home/santig14/fing/taa/1_TallerAA/50. preprocesamiento/results/10_p0vsp1vsp2vsp3vsp4_comparison.csv


In [6]:
p03_group_comparison = pd.DataFrame(p03_group_rows).sort_values("pipeline").reset_index(drop=True)
p03_group_path = results_dir / "10_p03_group_comparison.csv"
p03_group_comparison.to_csv(p03_group_path, index=False)
display(p03_group_comparison[columns].round(6))
print(f"Resultados por grupo P03 guardados en: {p03_group_path}")


,pipeline,best_ams,roc_auc,best_threshold,precision_at_ams,recall_at_ams,f1_at_ams,predicted_signal_count,feature_count,fit_seconds
0,P03_G0,0.738753,0.891660,0.855209,0.873457,0.339329,0.488774,972,20,1.291575
1,P03_G1,0.537608,0.890635,0.916172,0.943372,0.261786,0.409841,777,23,1.592016
2,P03_G2,0.607209,0.929319,0.888716,0.949387,0.568760,0.711358,1956,31,1.646850


Resultados por grupo P03 guardados en: /home/santig14/fing/taa/1_TallerAA/50. preprocesamiento/results/10_p03_group_comparison.csv


## Lectura rapida

La tabla principal ordena por `best_ams`. Si un preprocesamiento nuevo no supera de forma clara al baseline con este arbol fijo, no conviene tunearlo todavia. Si alguno supera claramente al resto, el siguiente paso razonable es repetir con validacion cruzada o hacer una busqueda de hiperparametros especifica para ese preprocesamiento.